In [1]:
import numpy as np


# Step 1: Define Corpus
text_data = "Data science and machine learning are powerful and machine learning is evolving"
tokens = text_data.lower().split()


# Step 2: Create Vocabulary
unique_words = sorted(set(tokens))
v_size = len(unique_words)

word_index = {w: idx for idx, w in enumerate(unique_words)}
index_word = {idx: w for w, idx in word_index.items()}

# One-hot encoding function
def to_one_hot(word):
    vec = np.zeros(v_size)
    vec[word_index[word]] = 1
    return vec

In [2]:
# Step 3: Prepare CBOW Dataset
window = 1
inputs = []
targets = []

for i in range(window, len(tokens) - window):
    context_words = []

    for j in range(i - window, i + window + 1):
        if j != i:
            context_words.append(to_one_hot(tokens[j]))

    avg_context = np.mean(context_words, axis=0)
    inputs.append(avg_context)
    targets.append(to_one_hot(tokens[i]))

inputs = np.array(inputs)
targets = np.array(targets)

In [3]:
# Step 4: Initialize Parameters
embed_dim = 8
W_input = np.random.uniform(-1, 1, (v_size, embed_dim))
W_output = np.random.uniform(-1, 1, (embed_dim, v_size))

lr = 0.05
epochs = 800

# Softmax function
def softmax_fn(x):
    exp_vals = np.exp(x - np.max(x))
    return exp_vals / np.sum(exp_vals)


In [4]:
# Step 5: Training Loop
for ep in range(epochs):
    total_loss = 0

    for i in range(len(inputs)):
        hidden_layer = np.dot(inputs[i], W_input)
        output_layer = np.dot(hidden_layer, W_output)

        prediction = softmax_fn(output_layer)

        # Cross-entropy loss
        loss = -np.sum(targets[i] * np.log(prediction + 1e-9))
        total_loss += loss

        # Backpropagation
        error = prediction - targets[i]

        grad_W_output = np.outer(hidden_layer, error)
        grad_W_input = np.outer(inputs[i], np.dot(W_output, error))

        W_output -= lr * grad_W_output
        W_input -= lr * grad_W_input

    if ep % 200 == 0:
        print(f"Epoch {ep} -> Loss: {total_loss:.4f}")

Epoch 0 -> Loss: 23.4848
Epoch 200 -> Loss: 0.1878
Epoch 400 -> Loss: 0.0733
Epoch 600 -> Loss: 0.0436


In [5]:
# Step 6: Extract Embeddings
word_vectors = W_input

print("\nVector representation of 'machine':")
print(word_vectors[word_index["machine"]])


# Step 7: Cosine Similarity
def similarity(w1, w2):
    v1 = word_vectors[word_index[w1]]
    v2 = word_vectors[word_index[w2]]

    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)

print("\nSimilarity (machine, learning):", similarity("machine", "learning"))
print("Similarity (data, science):", similarity("data", "science"))


# Step 8: Predict Target Word
def predict_word(context_list):
    ctx_vecs = [to_one_hot(w) for w in context_list]
    avg_vec = np.mean(ctx_vecs, axis=0)

    hidden = np.dot(avg_vec, W_input)
    output = np.dot(hidden, W_output)
    probs = softmax_fn(output)

    return index_word[np.argmax(probs)]

test_context = ["machine", "learning"]
print("\nContext:", test_context)
print("Predicted Word:", predict_word(test_context))


Vector representation of 'machine':
[-2.36870396  0.82037871  0.86534701 -1.61191088  1.22928091  0.12651581
  2.03164872 -0.65689302]

Similarity (machine, learning): -0.1850868027277241
Similarity (data, science): 0.14997330157156005

Context: ['machine', 'learning']
Predicted Word: and
